In [1]:
import amulet
import os
import sys
import json
import numpy as np
import pandas as pd
from amulet import load_level
from amulet_nbt import load
from amulet.api.selection import SelectionBox

import mcschematic

INFO - PyMCTranslate Version 385


In [2]:
data_path = '../data/'
processed_builds_path = os.path.join(data_path, 'processed_builds/')
new_data_path = os.path.join(data_path, 'new_data/')
java_data_path = os.path.join(new_data_path, 'java/')
universal_data_path = os.path.join(new_data_path, 'universal/')

schem_files = os.listdir(processed_builds_path)
files = [i for i in schem_files if i.endswith('.schematic') or i.endswith('.schem')]

In [3]:
def array_to_schematic(token_array, tok2block, save_path, filename="my_schematic"):
    """
    Converts a 3D numpy array of integer tokens into a Minecraft schematic file.

    Args:
        token_array (np.ndarray): A 3D array where values correspond to block types.
        filename (str): The name of the output schematic file (without extension).
    """
    schem = mcschematic.MCSchematic()

    d, h, w = token_array.shape

    # Iterate through the 3D array and place blocks
    for x in range(d):
        for y in range(h):
            for z in range(w):
                token = token_array[x, y, z]
                # block_name = tok2block.get(f'{token}', "minecraft:air") # Default to air if token not found
                block_name = tok2block[token]
                

                # Place the block in the schematic at the specified coordinates (x, y, z)
                schem.setBlock((x, y, z), block_name)

    # Save the schematic file
    schem.save(save_path, filename, mcschematic.Version.JE_1_21_5, True)
    print(f"Successfully saved schematic to {filename}.schem")

In [ ]:
# Initialize stuff
columns = [ "filename", "platform", "version", "x", "y", "z", "volume", "dims", "bounds", "error" ]

df_path = os.path.join(data_path, 'new_samples_metadata.csv')
universal_palette_path = os.path.join(data_path, 'universal_palette.json')
java_palette_path = os.path.join(data_path, 'java_palette.json')

# Load from checkpoint or initialize
checkpoint = 0
save_checkpoints = 100
volume_limit = 3000000

if checkpoint > 0:
    schem_df = pd.read_csv(df_path, index_col=0)
    
    # with open(universal_palette_path, 'r') as file:
    #     universal_palette = json.load(file)
    
    with open(java_palette_path, 'r') as file:
        java_palette = json.load(file)
    print(f'Resuming from schematic {checkpoint}')
else:
    schem_df = pd.DataFrame(columns=columns)
    
    # universal_palette = {}

    java_palette = {}
    
# Start looping through the .schem files
for i, file in enumerate(files):
    if i < checkpoint:
        continue
    
    row = {k: "" for k in columns}
    
    base_name = file.removesuffix('.schem')
    row["filename"] = base_name
    level_loaded = False
    try:
        schem_path = os.path.join(processed_builds_path, file)
        
        with open(schem_path, 'rb') as f:
            nbt = load(f, compressed=True)

            root = nbt.tag  # <-- THIS is the correct root (CompoundTag)

            w = root["Width"].py_int
            h = root["Height"].py_int
            l = root["Length"].py_int
            
            volume = w * h * l
        
        if volume > volume_limit:
            raise Exception(f"exceeded volume limit of {volume_limit}: {volume} from {w} x {h} x {l}")
        
        level = load_level(schem_path)
        level_loaded = True
        
        row["platform"] = level.level_wrapper.platform
        row["version"] = level.level_wrapper.version
        
        dim = level.dimensions[0]
        row["dims"] = str(level.dimensions)
        
        bounds = level.bounds(dim).bounds_array
        row["bounds"] = str(level.bounds(dim).bounds)
        
        dims = [ bounds[1,i] - bounds[0,i] for i in range(3) ]
        row["x"] = dims[0]
        row["y"] = dims[1]
        row["z"] = dims[2]
        row["volume"] = row["x"] * row["y"] * row["z"]
        
        # universal_array = np.zeros(dims, dtype=np.int16)
        java_array = np.zeros(dims, dtype=np.int16)
        
        translator = level.translation_manager.get_version("java", (1, 21, 5))
        
        for x in range(dims[0]):
            for y in range(dims[1]):
                for z in range(dims[2]):
                    # Get the block string
                    # universal_block = level.get_block( x, y, z, dim)
                    java_block = level.get_version_block( x, y, z, dim, ('java', (1, 19, 4)))[0]
                    # java_block, _, _ = translator.block.from_universal(universal_block, None)
                    
                    # universal_block_str = str(universal_block)
                    java_block_str = str(java_block.blockstate)
                    
                    # Add to palette if it isn't already
                    # if universal_block_str not in universal_palette.keys():
                    #     universal_palette[universal_block_str] = len(list(universal_palette.keys()))
                    
                    if java_block_str not in java_palette.keys():
                        java_palette[java_block_str] = len(list(java_palette.keys()))
                    
                    # Set this voxel in the build arrays
                    # universal_array[x,y,z] = universal_palette[universal_block_str]
                    java_array[x,y,z] = java_palette[java_block_str]
        
        # Create file paths for saving the universal and java format arrays
        # universal_save_path = os.path.join(universal_data_path, f'{base_name}_universal.npy')
        java_save_path = os.path.join(java_data_path, f'{base_name}_java.npy')
        
        # np.save(universal_save_path, universal_array)
        np.save(java_save_path, java_array)
            
        print(f'{i}\t: Processed {base_name}')
        
        if i % save_checkpoints == 0:
            tok2block = {v:k for k,v in java_palette.items()}
            array_to_schematic(java_array, tok2block, new_data_path, base_name)
        
    except Exception as e:
        row["error"] = str(e)
        print(f'{i}\t: Failed {base_name}: {str(e)}')
    finally:
        if level_loaded: level.close()
        
    schem_df.loc[len(schem_df)] = row
    
    if i % save_checkpoints == 0:
        # with open(universal_palette_path, 'w') as f:
        #     json.dump(universal_palette, f)
        
        with open(java_palette_path, 'w') as f:
            json.dump(java_palette, f)
        
        schem_df.to_csv(df_path, index=True)
        
        print(f'{i}\t: Saved Metadata & build sample')
        

INFO - Loading level ../data/processed_builds/build_batch_100_2574_1.schem
INFO - Loading level ../data/processed_builds/build_batch_100_2578_1.schem


0	: Processed build_batch_100_2574_1
Successfully saved schematic to build_batch_100_2574_1.schem
closed level
0	: Saved Metadata & build sample


WARNING - Could not find translation information for block minecraft:grass_path to universal in PyMCTranslate.Version(java, (1, 20, 1)). If this is not a vanilla block ignore this message
WARNING - Could not find translation information for block minecraft:grass_path from universal in PyMCTranslate.Version(java, (1, 19, 4)). If this is not a vanilla block ignore this message
INFO - Loading level ../data/processed_builds/build_batch_100_2592_1.schem


1	: Processed build_batch_100_2578_1
closed level
2	: Failed build_batch_100_2587_1: exceeded volume limit of 3000000: 12902400 from 448 x 72 x 400
closed level
closed level


KeyboardInterrupt: 